In [0]:
%run "../00_common/01_environment_config"

In [0]:
%run "../00_common/02_bronze_helpers"

In [0]:
print(landing_folder_path)
print(add_ingestion_metadata)

In [0]:
# Databricks notebook source

# ============================================================
# F1 PROJECT - INGEST RACES FILE
# ============================================================
# Purpose:
# 1. Read races.csv using Spark DataFrame Reader API
# 2. Apply an explicit schema
# 3. Add ingestion metadata
# 4. Write the data into a Bronze Delta table
# ============================================================



# COMMAND ----------

# MAGIC %run "../00_common/01_environment_config"


# COMMAND ----------

# MAGIC %run "../00_common/02_bronze_helpers"


# COMMAND ----------

# Source file location

source_file = f"{landing_folder_path}/races.csv"

# Target Bronze table

table_name = (
    f"{catalog_name}.{schema_name}.{bronze_prefix}_races"
)

print("Source File :", source_file)
print("Target Table:", table_name)


# COMMAND ----------

# ============================================================
# Step 1 - Define schema for races.csv
# ============================================================

from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    DateType
)

races_schema = StructType([
    StructField("season", IntegerType(), True),
    StructField("round", IntegerType(), True),
    StructField("url", StringType(), True),
    StructField("raceName", StringType(), True),
    StructField("date", DateType(), True),
    StructField("circuitId", StringType(), True)
])


# COMMAND ----------

# ============================================================
# Step 2 - Read races.csv
# ============================================================

races_df = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("mode", "FAILFAST")
    .option("dateFormat", "yyyy-MM-dd")
    .schema(races_schema)
    .load(source_file)
)

display(races_df)


# COMMAND ----------

# Check DataFrame schema

races_df.printSchema()


# COMMAND ----------

# ============================================================
# Step 3 - Add ingestion metadata
# ============================================================

races_final_df = add_ingestion_metadata(races_df)

display(races_final_df)


# COMMAND ----------

# ============================================================
# Step 4 - Write to Bronze Delta table
# ============================================================

(
    races_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(table_name)
)


# COMMAND ----------

# ============================================================
# Step 5 - Validate Bronze Delta table
# ============================================================

display(
    spark.table(table_name)
)